# SafeStep — Colab 실행 노트북 (CUDA GPU용)
> ⚠️ **AI Hub 데이터에는 쓰지 마세요.** AI Hub는 해외 IP(Colab 포함) 다운로드를 차단하고, 약관상 국외 반출에 별도 합의가 필요합니다. AI Hub 데이터는 README의 **맥(MLX) 실행 방법**을 따르세요.
> 이 노트북은 직접 촬영한 영상처럼 반출 제한이 없는 데이터용입니다.

런타임 → 런타임 유형 변경 → **GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Drive 안에서 프로젝트 폴더(common.py + requirements.txt 가 있는 곳)를 자동으로 찾는다
import glob, os
hits = [os.path.dirname(p) for p in glob.glob('/content/drive/MyDrive/**/requirements.txt', recursive=True)
        if os.path.exists(os.path.join(os.path.dirname(p), '01b_import_images.py'))]
print('찾은 위치:', hits)
assert hits, 'Drive에서 safestep-vlm 폴더를 못 찾았습니다. zip을 풀어서 폴더째 업로드했는지 확인하세요.'
os.chdir(hits[0])
print('현재 위치:', os.getcwd())
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -r requirements.txt

## 0. (AI Hub 데이터 사용 시) 인도보행 영상 다운로드
사전 준비: aihub.or.kr 데이터셋 189 다운로드 **승인 완료** + API key 발급.
API key는 코드에 직접 쓰지 말고 Colab 왼쪽 🔑(Secrets)에 `AIHUB_KEY` 로 저장.
원본 zip은 Colab 로컬 디스크(/content/aihub)에 받고, 정리된 프레임만 Drive에 저장한다.

In [ ]:
!curl -s -o /content/aihubshell https://api.aihub.or.kr/api/aihubshell.do && chmod +x /content/aihubshell
!/content/aihubshell -mode l -datasetkey 189 | grep -i bbox | head

In [ ]:
# 다운로드 (10~30분). 진행 상황이 아래에 계속 출력되고, key는 *** 로 가려진다.
import os, subprocess
from google.colab import userdata
key = (userdata.get('AIHUB_KEY') or '').strip()
assert key, 'Secrets에 AIHUB_KEY가 없거나 노트북 액세스가 꺼져 있습니다.'
print('key 길이:', len(key))
os.makedirs('/content/aihub', exist_ok=True)
p = subprocess.Popen(['/content/aihubshell', '-mode', 'd', '-datasetkey', '189', '-filekey', '49933',
                      '-aihubapikey', key], cwd='/content/aihub',
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line.replace(key, '***'), end='')
print('\n종료 코드:', p.wait())

In [ ]:
# 다운로드 결과 확인: 폴더 구조 + 확장자별 파일 수 + 용량
!find /content/aihub -maxdepth 5 -type d | head -20
!find /content/aihub -type f | sed 's/.*\.//' | sort | uniq -c | sort -rn | head
!du -sh /content/aihub; df -h /content | tail -1

In [ ]:
# (이미지 없이 .zip / .zip.part* 만 있을 때만 실행) 직접 병합 + 압축 해제
import glob, os, re, subprocess
from collections import defaultdict
groups = defaultdict(list)
for f in glob.glob('/content/aihub/**/*.zip.part*', recursive=True):
    groups[f.split('.part')[0]].append(f)
for zp, parts in groups.items():
    parts.sort(key=lambda f: int(re.search(r'\.part(\d+)$', f).group(1)))
    print('병합:', zp, len(parts), '조각')
    with open(zp, 'wb') as out:
        for f in parts:
            with open(f, 'rb') as src:
                while chunk := src.read(64 << 20):
                    out.write(chunk)
    for f in parts: os.remove(f)
for zp in glob.glob('/content/aihub/**/*.zip', recursive=True):
    print('압축 해제:', zp)
    subprocess.run(['unzip', '-q', '-o', zp, '-d', os.path.dirname(zp)], check=True)
    os.remove(zp)
!find /content/aihub -type f | sed 's/.*\.//' | sort | uniq -c | sort -rn | head

In [ ]:
!python 01b_import_images.py --src /content/aihub --out data/frames --stride 10 --max-per-group 40 --max-total 1500
!ls data/frames | wc -l

## 1. 프레임 추출

In [ ]:
!python 01_extract_frames.py --videos data/raw_videos --out data/frames --interval 1.0 --blur-faces

## 2. YOLO + 깊이 힌트

In [ ]:
!python 02_aux_hints.py

## 3. Teacher 라벨링 (끊기면 이 셀만 다시 실행 → 이어서 진행)

In [ ]:
!python 03_teacher_label.py

## 4. 사람 검수 (출력되는 gradio.live 링크 클릭, 끝나면 셀 중지)

In [ ]:
!python 04_review_app.py --share

## 5. 분할 + 통계 (+ HF 업로드: `huggingface-cli login` 후 `--push-to-hub 아이디/safestep-ko`)

In [ ]:
!python 05_build_splits.py

## 6. QLoRA 학습

In [ ]:
!python 06_train_qlora.py --out outputs/lora-r16
from IPython.display import Image; Image('outputs/lora-r16/loss_curve.png')

## 7. 평가

In [ ]:
!python 07_evaluate.py run --name base3b
!python 07_evaluate.py run --name lora_r16 --adapter outputs/lora-r16/final
!python 07_evaluate.py run --name teacher7b --model Qwen/Qwen2.5-VL-7B-Instruct --max-pixels 602112
!python 07_evaluate.py table --names base3b lora_r16 teacher7b

## (선택) Ablation: rank / vision LoRA

In [ ]:
!python 06_train_qlora.py --rank 8 --alpha 16 --out outputs/lora-r8
!python 06_train_qlora.py --tune-vision --out outputs/lora-r16-vis
!python 07_evaluate.py run --name lora_r8 --adapter outputs/lora-r8/final
!python 07_evaluate.py run --name lora_r16_vis --adapter outputs/lora-r16-vis/final
!python 07_evaluate.py table --names base3b lora_r8 lora_r16 lora_r16_vis teacher7b

## 8. 데모

In [ ]:
!python 08_demo_app.py --adapter outputs/lora-r16/final --share